In [1]:
import pandas as pd
import sys
sys.path.append('/')
from utils_processing import transform_verbose_to_calendar, add_country_codes, split_been_planned, count_trips_per_year, split_lived_visited
from utils_plot import plot_bar_time_series, create_map, create_tree

TODAY = '2026-11-30'

In [2]:
df_verbose = pd.read_csv("../data/trips_verbose.csv", sep=None, engine="python")

# Split into been vs planned relative to today (straddling trips classified by larger portion)
df_verbose_been, df_verbose_planned = split_been_planned(df_verbose, today=TODAY)

df = transform_verbose_to_calendar(df_verbose_been, save=False)
df = df[df['country'] != 'All']

if len(df_verbose_planned) > 0:
    df_planned = transform_verbose_to_calendar(df_verbose_planned, save=False)
    df_planned = df_planned[df_planned['country'] != 'All']
else:
    df_planned = None

# Aggregate city-level rows to one row per country (year columns become days per country per year)
NUM_COLS = df.select_dtypes(include='number').columns.tolist()
df_country = df.groupby(['country','continent'])[NUM_COLS].sum().reset_index()
df_country = add_country_codes(df_country.copy())
df_country['total_years'] = df_country['total_days']/365

# Trips per year, split into domestic vs international (excursions away from
# that year's primary residence city; uses the city-level calendar df)
trip_counts = count_trips_per_year(df_verbose_been, df)

plot_bar_time_series(
    df_country, 
    select_type='travel', 
    time_period='year', 
    title='Been there', 
    trip_counts=trip_counts,
    save_path='../attachments/bar_time_series.html',
    #save_path=None
    )

In [3]:
RESIDENCE_OVERRIDES = [
    ('China',          '1996-10-04', '2015-08-18'),
    ('United States',  '2015-08-18', '2018-09-21'),
    ('United Kingdom', '2018-09-21', '2020-09-15'),
    ('Sweden',         '2020-09-15', None),  # None = ongoing
]
# RESIDENCE_OVERRIDES = None

COLOR_DICT = {'Asia':'gold', 'Europe':'royalblue', 'North America':'blueviolet', 'South America':'mediumseagreen', 'Africa':'crimson'}

def fmt_days(d):
    # Over a year: show years; under a day: one decimal; otherwise whole days
    if d > 365:
        return f'{d/365:.1f} years'
    if d < 1:
        return f'{d:.1f} days'
    n = round(d)
    return f'{n} day' if n == 1 else f'{n} days'

def fmt_pct(d, share):
    # Shown in years (over 365 days): whole percent; shown in days: one
    # decimal, or nothing if that would still read 0.0%
    if d > 365:
        return f'{share:.0%}'
    pct = f'{share:.1%}'
    return None if pct == '0.0%' else pct
FONT_SIZE_DICT = {'base':18, 'title':20, 'tree_text':18}

In [11]:
n_country = df_country.country.nunique()
t_country = df_country.total_days.sum()

create_tree(
    df_country,
    feat='country',
    var='total_days',
    flag='continent',
    group_flag=True,
    color_dict=COLOR_DICT,
    font_size_dict=FONT_SIZE_DICT,
    title=f'{n_country} countries, {fmt_days(t_country)} (log size)',
    threshold=6,
    threshold_global=False,
    expand_others=True,
    start_collapsed=True,
    log_scale=True,
    value_fmt=fmt_days,
    pct_fmt=fmt_pct,
    save_path='../attachments/tree_travel.html',
    #save_path=None
    )

In [5]:
# import pycountry
# # using the official name
# country = pycountry.countries.lookup("Vatican City State")
# print(country.alpha_2, country.alpha_3, country.name)

In [6]:
VAR = 'total_days'
# first item is the default
#LEVEL = {'city': 'city', 'country': 'country'}
LEVEL = 'city'
#LEVEL = 'country'
# first item is the default;
# orthographic, equirectangular, mercator, natural earth, kavrayskiy7, miller, robinson, eckert4, azimuthal equal area, azimuthal equidistant, gnomonic, stereographic, mollweide
#PROJECTION_TYPE = {'3D': 'orthographic', '2D': 'eckert4'}
PROJECTION_TYPE = {'3D': 'orthographic', '2D': 'mercator'}
BINS = [0, 7, 30, 365, df.groupby('country')[VAR].sum().max()]

create_map(
    df, 
    var=VAR, 
    code_convention='code3', 
    bins=BINS,
    color='royalblue',
    projection_type=PROJECTION_TYPE, 
    tooltip_mode='calendar',
    level=LEVEL, 
    df_planned=df_planned,
    save_path='../attachments/map_travel.html'
    #save_path=None
    )